# 多模态共享LoRA梯度冲突分析（完整梯度版）

你已补充 `grad` 字段（完整参数梯度信息），本Notebook在原有大文件优化基础上，新增：

1. `grad` 向量解析与缓存（支持 list / npy路径 / 字符串list）  
2. text/image 梯度**方向余弦相似度**（核心冲突证据）  
3. 子空间容量分析（SVD能量覆盖 rank）  
4. 层级冲突强度与容量瓶颈的联合可视化

> 仍保持按模块按需加载，避免一次性读完整日志导致内存压力。


In [ ]:
import ast
import json
import hashlib
from pathlib import Path
from typing import Callable, Iterable, Optional

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="talk")
pd.set_option('display.max_columns', 200)


In [ ]:
# ===== 配置区 =====
LOG_PATH = Path('gradient_logs.jsonl')
OUTPUT_DIR = Path('analysis_outputs')
CACHE_DIR = OUTPUT_DIR / 'vector_cache'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 大文件快速调试开关
STEP_MIN = None
STEP_MAX = None
STEP_MOD = None   # 如 5 表示每5步取1步

# 向量分析抽样（防止一次处理过大）
MAX_VECTOR_ROWS = 50000
MAX_PAIR_ROWS = 20000

assert LOG_PATH.exists(), f'文件不存在: {LOG_PATH.resolve()}'
print('Using log file:', LOG_PATH.resolve())


In [ ]:
def _parse_modality_ids(x):
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            y = json.loads(x)
            return y if isinstance(y, list) else []
        except Exception:
            try:
                y = ast.literal_eval(x)
                return y if isinstance(y, list) else []
            except Exception:
                return []
    return []


def _step_pass(step_val, step_min=None, step_max=None, step_mod=None):
    try:
        s = int(step_val)
    except Exception:
        return False
    if step_min is not None and s < step_min:
        return False
    if step_max is not None and s > step_max:
        return False
    if step_mod is not None and step_mod > 1 and (s % step_mod != 0):
        return False
    return True


def load_jsonl_subset(
    path: Path,
    usecols: Optional[Iterable[str]] = None,
    partition_in: Optional[set] = None,
    step_min=None,
    step_max=None,
    step_mod=None,
    row_predicate: Optional[Callable[[dict], bool]] = None,
    max_rows: Optional[int] = None,
):
    rows = []
    usecols = set(usecols) if usecols is not None else None

    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            step_val = obj.get('step', None)
            if step_val is not None and not _step_pass(step_val, step_min, step_max, step_mod):
                continue

            if partition_in is not None and obj.get('grad_partition', 'all') not in partition_in:
                continue

            if row_predicate is not None and not row_predicate(obj):
                continue

            row = obj if usecols is None else {k: obj.get(k, None) for k in usecols}
            rows.append(row)
            if max_rows is not None and len(rows) >= max_rows:
                break

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    if 'modality_ids' in df.columns:
        df['modality_ids'] = df['modality_ids'].apply(_parse_modality_ids)

    for c in ['step','layer_depth','grad_norm','grad_mean','grad_std','grad_abs_mean','numel','supervised_token_count']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    if 'grad_was_none' in df.columns:
        df['grad_was_none'] = df['grad_was_none'].fillna(False).astype(bool)

    if 'grad_partition' in df.columns:
        df['grad_partition'] = df['grad_partition'].fillna('all')

    return df


## 一、轻量基础分析（沿用）

先做不依赖完整向量的统计图，快速看趋势。

In [ ]:
df_meta = load_jsonl_subset(
    LOG_PATH,
    usecols=['step','grad_partition','layer_depth','param_name','grad_was_none','grad_norm'],
    step_min=STEP_MIN, step_max=STEP_MAX, step_mod=STEP_MOD,
)

summary = {
    'rows_loaded': len(df_meta),
    'step_min': int(df_meta['step'].min()) if len(df_meta)>0 else None,
    'step_max': int(df_meta['step'].max()) if len(df_meta)>0 else None,
    'n_layers': int(df_meta['layer_depth'].nunique()) if 'layer_depth' in df_meta else 0,
    'n_params': int(df_meta['param_name'].nunique()) if 'param_name' in df_meta else 0,
    'partitions': sorted(df_meta['grad_partition'].dropna().unique().tolist()) if 'grad_partition' in df_meta else [],
    'none_ratio': float(df_meta['grad_was_none'].mean()) if 'grad_was_none' in df_meta and len(df_meta)>0 else None,
}
summary


In [ ]:
if len(df_meta) > 0:
    plt.figure(figsize=(8,4))
    sns.countplot(data=df_meta, x='grad_partition', order=[p for p in ['all','text_only','image_only'] if p in df_meta['grad_partition'].unique()])
    plt.title('Partition Record Count')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/'fig_partition_count.png', dpi=180); plt.show()

    trend = df_meta.groupby(['step','grad_partition'], as_index=False)['grad_norm'].mean()
    plt.figure(figsize=(12,5))
    sns.lineplot(data=trend, x='step', y='grad_norm', hue='grad_partition', linewidth=2)
    plt.title('Mean Grad Norm over Steps by Partition')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/'fig_step_partition_gradnorm_trend.png', dpi=180); plt.show()


## 二、完整梯度字段解析（新增核心）

支持以下 `grad` 存储形式：

- Python list（日志直接存数组）
- 字符串化 list（如 `"[0.1, ...]"`）
- `.npy` 路径字符串

并将向量缓存为 `vector_cache/*.npy`，后续复用避免重复解析。


In [ ]:
def _stable_hash(text: str) -> str:
    return hashlib.md5(text.encode('utf-8')).hexdigest()


def parse_grad_to_npy(grad_field, cache_dir: Path):
    if grad_field is None:
        return None

    # case 1: already list/ndarray
    if isinstance(grad_field, (list, tuple, np.ndarray)):
        arr = np.asarray(grad_field, dtype=np.float32).reshape(-1)
        key = _stable_hash(str(arr.shape) + ':' + str(float(arr.sum())))
        out = cache_dir / f'{key}.npy'
        if not out.exists():
            np.save(out, arr)
        return str(out)

    # case 2: string
    if isinstance(grad_field, str):
        g = grad_field.strip()
        if not g:
            return None
        # npy path
        if g.endswith('.npy') and Path(g).exists():
            return g
        # json / literal list
        try:
            obj = json.loads(g)
            if isinstance(obj, list):
                arr = np.asarray(obj, dtype=np.float32).reshape(-1)
                key = _stable_hash(g[:200] + f'|len={len(arr)}')
                out = cache_dir / f'{key}.npy'
                if not out.exists():
                    np.save(out, arr)
                return str(out)
        except Exception:
            pass
        try:
            obj = ast.literal_eval(g)
            if isinstance(obj, (list, tuple)):
                arr = np.asarray(obj, dtype=np.float32).reshape(-1)
                key = _stable_hash(g[:200] + f'|len={len(arr)}')
                out = cache_dir / f'{key}.npy'
                if not out.exists():
                    np.save(out, arr)
                return str(out)
        except Exception:
            return None
    return None


def load_grad_vectors_df(max_rows=MAX_VECTOR_ROWS):
    df = load_jsonl_subset(
        LOG_PATH,
        usecols=['step','param_name','layer_depth','adapter_type','grad_partition','grad','grad_norm'],
        partition_in={'all','text_only','image_only'},
        step_min=STEP_MIN, step_max=STEP_MAX, step_mod=STEP_MOD,
        max_rows=max_rows,
    )
    if len(df) == 0:
        return df

    if 'grad' not in df.columns:
        print('[WARN] 日志中没有 grad 字段，无法进行向量分析。')
        return pd.DataFrame()

    df['grad_path'] = df['grad'].apply(lambda x: parse_grad_to_npy(x, CACHE_DIR))
    df = df[df['grad_path'].notna()].copy()
    return df


In [ ]:
df_vec = load_grad_vectors_df()
print('vector rows:', len(df_vec))
df_vec.head(3)


## 三、方向冲突分析：cos(text, image)

这部分是你论文3.2.1最关键证据。

In [ ]:
def cosine(a, b, eps=1e-12):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps))

if len(df_vec) > 0:
    sub = df_vec[df_vec['grad_partition'].isin(['text_only','image_only'])].copy()
    key = ['step','param_name','layer_depth','adapter_type']

    txt = sub[sub['grad_partition']=='text_only'][key+['grad_path']].rename(columns={'grad_path':'text_path'})
    img = sub[sub['grad_partition']=='image_only'][key+['grad_path']].rename(columns={'grad_path':'image_path'})
    pair = txt.merge(img, on=key, how='inner').head(MAX_PAIR_ROWS)

    cos_vals = []
    for _, r in pair.iterrows():
        vt = np.load(r['text_path']).reshape(-1)
        vi = np.load(r['image_path']).reshape(-1)
        m = min(len(vt), len(vi))
        if m == 0:
            cos_vals.append(np.nan)
            continue
        cos_vals.append(cosine(vt[:m], vi[:m]))

    pair['cosine_text_image'] = cos_vals
    pair = pair.dropna(subset=['cosine_text_image'])

    plt.figure(figsize=(9,4))
    sns.histplot(pair['cosine_text_image'], bins=60, kde=True)
    plt.axvline(0, color='red', linestyle='--')
    plt.title('Cosine(text_grad, image_grad)')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/'fig_cosine_hist_text_image.png', dpi=180); plt.show()

    layer_cos = pair.groupby('layer_depth', as_index=False)['cosine_text_image'].mean().sort_values('layer_depth')
    plt.figure(figsize=(11,5))
    sns.lineplot(data=layer_cos, x='layer_depth', y='cosine_text_image', marker='o')
    plt.axhline(0, color='red', linestyle='--')
    plt.title('Layer-wise Mean Cosine(text,image)')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/'fig_layer_cosine_text_image.png', dpi=180); plt.show()

    display(pair['cosine_text_image'].describe())
else:
    print('No vectorized grad data found.')


## 四、子空间容量分析（SVD）

对应你第3.3节：比较 text/image 所需主成分rank（95%能量覆盖）。

In [ ]:
def svd_energy_rank(X, energy=0.95):
    # X: [n_samples, d]
    if X.shape[0] < 2:
        return np.nan
    X = X - X.mean(axis=0, keepdims=True)
    s = np.linalg.svd(X, full_matrices=False, compute_uv=False)
    e = (s ** 2)
    cum = np.cumsum(e) / (e.sum() + 1e-12)
    r = int(np.searchsorted(cum, energy) + 1)
    return r

if len(df_vec) > 0:
    # 为控制计算量，按层每个分区最多取N条
    MAX_PER_LAYER_PART = 200
    ranks = []

    for layer, g1 in df_vec.groupby('layer_depth'):
        for part in ['text_only', 'image_only', 'all']:
            g2 = g1[g1['grad_partition'] == part].head(MAX_PER_LAYER_PART)
            if len(g2) < 2:
                continue
            vecs = [np.load(p).reshape(-1) for p in g2['grad_path'].tolist()]
            dmin = min(len(v) for v in vecs)
            if dmin <= 1:
                continue
            X = np.stack([v[:dmin] for v in vecs], axis=0)
            r95 = svd_energy_rank(X, energy=0.95)
            r90 = svd_energy_rank(X, energy=0.90)
            ranks.append({'layer_depth': layer, 'grad_partition': part, 'rank90': r90, 'rank95': r95, 'n_samples': len(vecs), 'dim_used': dmin})

    rank_df = pd.DataFrame(ranks)
    if len(rank_df) > 0:
        plt.figure(figsize=(11,5))
        sns.lineplot(data=rank_df, x='layer_depth', y='rank95', hue='grad_partition', marker='o')
        plt.title('Layer-wise Rank@95% Energy by Partition')
        plt.tight_layout(); plt.savefig(OUTPUT_DIR/'fig_layer_rank95_partition.png', dpi=180); plt.show()

        display(rank_df.head(10))
        rank_df.to_csv(OUTPUT_DIR/'table_layer_svd_rank.csv', index=False, encoding='utf-8-sig')
    else:
        print('Not enough vector samples for SVD rank analysis.')


## 五、冲突-容量联合图（论文强图建议）

将层级余弦（冲突）与 rank95（容量）放在一起，直接支撑“冲突 + 容量不足”论点。

In [ ]:
if 'pair' in globals() and 'rank_df' in globals() and isinstance(rank_df, pd.DataFrame) and len(rank_df) > 0:
    layer_cos = pair.groupby('layer_depth', as_index=False)['cosine_text_image'].mean()
    rank_pivot = rank_df.pivot_table(index='layer_depth', columns='grad_partition', values='rank95', aggfunc='mean').reset_index()

    mix = layer_cos.merge(rank_pivot, on='layer_depth', how='inner')
    if len(mix) > 0:
        fig, ax1 = plt.subplots(figsize=(12,5))
        ax1.plot(mix['layer_depth'], mix['cosine_text_image'], color='tab:red', marker='o', label='cos(text,image)')
        ax1.axhline(0, color='tab:red', linestyle='--', alpha=0.7)
        ax1.set_ylabel('mean cosine', color='tab:red')
        ax1.set_xlabel('layer_depth')

        ax2 = ax1.twinx()
        if 'text_only' in mix.columns:
            ax2.plot(mix['layer_depth'], mix['text_only'], color='tab:blue', marker='^', label='rank95 text')
        if 'image_only' in mix.columns:
            ax2.plot(mix['layer_depth'], mix['image_only'], color='tab:green', marker='s', label='rank95 image')
        ax2.set_ylabel('rank95', color='tab:blue')

        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')
        plt.title('Conflict-Capacity Joint View by Layer')
        plt.tight_layout(); plt.savefig(OUTPUT_DIR/'fig_joint_conflict_capacity.png', dpi=180); plt.show()
    else:
        print('No overlap between cosine and rank tables.')
else:
    print('Need cosine and rank_df results first.')


## 六、导出论文表格

导出：
- `table_layer_cosine.csv`
- `table_layer_svd_rank.csv`
- `table_pair_cosine_summary.csv`


In [ ]:
if 'pair' in globals() and isinstance(pair, pd.DataFrame) and len(pair) > 0:
    layer_cos_tbl = pair.groupby('layer_depth', as_index=False)['cosine_text_image'].agg(['mean','std','count']).reset_index()
    layer_cos_tbl.columns = ['layer_depth','cosine_mean','cosine_std','n']
    layer_cos_tbl.to_csv(OUTPUT_DIR/'table_layer_cosine.csv', index=False, encoding='utf-8-sig')

    pair[['cosine_text_image']].describe().to_csv(OUTPUT_DIR/'table_pair_cosine_summary.csv', encoding='utf-8-sig')
    print('Saved cosine tables.')

if 'rank_df' in globals() and isinstance(rank_df, pd.DataFrame) and len(rank_df) > 0:
    rank_df.to_csv(OUTPUT_DIR/'table_layer_svd_rank.csv', index=False, encoding='utf-8-sig')
    print('Saved SVD rank table.')


## 七、结论模板（可直接写论文）

1. 若 `cosine(text,image)` 在中低层显著小于0：说明方向冲突直接存在。  
2. 若余弦接近0但rank95明显偏高：说明方向独立+共享容量竞争。  
3. 若 image_only 的rank95长期高于text_only：说明视觉子空间需求更高。  
4. 联合图中“低余弦 + 高rank”层可作为第4章结构改造重点层。
